# MISR to AT3D Preprocessing of the observations (BRF and geometry)

**Warning:** It is important that cells be run in order and without duplication. Certain transformations modify variables in place and applying the transformation twice will yield unexpected behavior.

After MISR data has been reformatted into the custom xarray format (i.e. `MISR_preprocessA.ipynb`), the data are processed in this notebook. The steps that take place here are:

1. Azimuth coordinate transformation
2. SOM x and y coordinate transformation
3. Merge position and view geometry ray-by-ray
4. Perform wind correction
5. Project imagery to the top of a radiative transfer grid
6. Convert $BRF$ to $I^*$

In [1]:
# conda env 'at3d_pure_k9'
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import at3d

PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

from src.utils.util import save_sensor_dict
import src.CoordinateTransform as ct

In [2]:
orbit = 105254
path = 117
imagery = xr.load_dataset(PROJECT_ROOT / 'data' / 'interim' / f'MISR_RAD_O{orbit:06d}_P{path:03d}_R00275_X17950487_18003012_Y0193187_0245712.nc')
gmp = xr.load_dataset(PROJECT_ROOT / 'data' / 'interim' / f'MISR_GEO_O{orbit:06d}_P{path:03d}_R17600_X17959150_17994350_Y0201850_0237050.nc')

## 1. Azimuth angle coordiante transform

Convert from azimuth relative to local north going clockwise to azimuth angles relative to local east going counter-clockwise.

$\phi_E = (90 - \phi_N) \mod 360$

Such that $\phi_N$ is the azimuthal angle (degrees) relative to local north going clockwise, and $\phi_E$ is the new azimuthal angle (degrees) relative to local east going counter-clockwise.

In [3]:
for var in gmp.data_vars:
    if('Azimuth' in var):
        gmp[var] = (90 - gmp[var]) % 360
        gmp[var].attrs['description'] = 'Azimuth relative to local East going counter-clockwise'

## 2. SOM x and y coordinate transform

In [4]:
origin = gmp.isel(som_x=1, som_y=1)
transform = ct.CoordinateTransform(
    origin.som_x.data,
    origin.som_y.data,
    origin.gamma.data,
    17600,
    17600
)
print(transform)

X, Y = np.meshgrid(imagery.som_x, imagery.som_y, indexing='ij')
Xp, Yp = transform.SOM2AT3D(X, Y)

imagery['x'] = (['som_x', 'som_y'], Xp)
imagery['y'] = (['som_x', 'som_y'], Yp)
imagery['z'] = (['som_x', 'som_y'], np.zeros_like(X))

Domain center som x: 17976750.0
Domain center som y: 219450.0
AT3D domain length x: 17600
AT3D domain length y: 17600
gamma: 1.655084388740932 radians


In [5]:
# ## Create AT3D coordinates
# origin = gmp.isel(som_x=1, som_y=1)
# X, Y = np.meshgrid(imagery.som_x, imagery.som_y, indexing='ij')
# ## 1. Translation setting origin to domain center
# Xt = X - origin.som_x.data
# Yt = Y - origin.som_y.data
# # 2. Rotation about the origin orienting with north
# Xtr = Xt * np.cos(origin.gamma.data) + Yt * np.sin(origin.gamma.data)
# Ytr = -Xt * np.sin(origin.gamma.data) + Yt * np.cos(origin.gamma.data)
# ## 3. Translation setting corner of domain
# Xtrt = Xtr + 8800
# Ytrt = Ytr + 8800

# imagery['x'] = (['som_x', 'som_y'], Xtrt)
# imagery['y'] = (['som_x', 'som_y'], Ytrt)
# imagery['z'] = (['som_x', 'som_y'], np.zeros_like(X))

## 3. Merge position (x, y, z) and viewing geometry

In [6]:
phi = np.array([origin[f'{c.lower().capitalize()}Azimuth'].data for c in imagery.camera.data])
mu = np.array([origin[f'{c.lower().capitalize()}Zenith'].data for c in imagery.camera.data])

In [7]:
print(imagery.camera.data)
print(phi)
print(mu)

['DF' 'CF' 'BF' 'AF' 'AN' 'AA' 'BA' 'CA' 'DA']
[ 72.019944    70.45855356  67.62234824  60.37507497 347.75077605
 269.34135341 261.85587018 259.02491631 257.55655947]
[70.42228051 60.33261609 46.15332394 27.1948554   7.21664647 26.57757211
 45.67785273 60.06890652 70.35949442]


In [8]:
tmp_shape = (imagery.sizes['camera'], imagery.sizes['som_x'], imagery.sizes['som_y'])
imagery['x'] = (['camera', 'som_x', 'som_y'], np.broadcast_to(imagery.x.data, tmp_shape))
imagery['y'] = (['camera', 'som_x', 'som_y'], np.broadcast_to(imagery.y.data, tmp_shape))
imagery['z'] = (['camera', 'som_x', 'som_y'], np.broadcast_to(imagery.z.data, tmp_shape))

phi = np.array([origin[f'{c.lower().capitalize()}Azimuth'].data for c in imagery.camera.data])
phi = np.radians(phi)
imagery['phi'] = (['camera', 'som_x', 'som_y'], np.broadcast_to(phi[:, None, None], tmp_shape))

mu = np.array([origin[f'{c.lower().capitalize()}Zenith'].data for c in imagery.camera.data])
mu = np.cos(np.radians(mu))
imagery['mu'] = (['camera', 'som_x', 'som_y'], np.broadcast_to(mu[:, None, None], tmp_shape))

## 4. Wind correction

In [9]:
u = -0.2 # m/s
v = 2.3 # m/s

imagery['deltat'] = (['camera'], np.array([-204, -144, -91, -45, 0, 45, 91, 144, 204]))

imagery['x'] = imagery.x - u*imagery.deltat
imagery['y'] = imagery.y - v*imagery.deltat

## 6. Create AT3D sensor objects

- Transform each ray to top of radiative transfer domain
- Convert $R$ to $I^*$

$I^* = \frac{R}{\pi}$

Where $R$ is the MISR BRF and $\mu_0$ is the cosine of the solar zenith angle.

In [10]:
def register_to_height(x, y, z, mu, phi, h):
    """
    Transform x, y coordinates to be registered at height h.
    
    Parameters:
    -----------
    x, y, z : array-like
        Original coordinates
    mu : array-like
        Cosine of zenith angle
    phi : array-like
        Azimuthal angle in radians
    h : float
        Target height for registration
        
    Returns:
    --------
    x_new, y_new : arrays
        Transformed coordinates at height h
    """
    # Calculate the distance to travel along the viewing direction
    # to reach height h
    dz = h - z
    
    # sin(zenith) = sqrt(1 - mu^2)
    sin_zenith = np.sqrt(1 - mu**2)
    
    # Displacement in x and y directions
    dx = (dz / mu) * sin_zenith * np.cos(phi)
    dy = (dz / mu) * sin_zenith * np.sin(phi)
    
    # New coordinates
    x_new = x + dx
    y_new = y + dy
    z_new = h * np.ones_like(z)
    
    return x_new, y_new, z_new

In [11]:
import numpy as np


def filter_rays_bbox(
    x, y, z, mu, phi,
    xmin, xmax,
    ymin, ymax,
    zmin, zmax,
):
    """
    Return a boolean mask selecting rays that intersect an axis-aligned
    bounding box (AABB).

    Ray parametric form:
        P(t) = origin + t * direction
    where direction is derived from the physics convention:
        mu  – cosine of the polar angle θ  (mu = cos θ)
        phi – azimuthal angle φ  (radians)

    So the unit-direction components are:
        dx = sqrt(1 - mu²) * cos(phi)
        dy = sqrt(1 - mu²) * sin(phi)
        dz = mu

    The slab method computes, for each axis, the interval [t_near, t_far]
    at which the ray is inside that axis's slab.  A ray hits the box when
    the intersection of all three intervals is non-empty:
        max(t_near_x, t_near_y, t_near_z) <= min(t_far_x, t_far_y, t_far_z)
    and that overlap has t_far >= 0  (box is not entirely behind the origin).

    Parameters
    ----------
    x, y, z : array-like, shape (N,)
        Ray origin coordinates.
    mu : array-like, shape (N,)
        Cosine of the polar angle (−1 ≤ mu ≤ 1).
    phi : array-like, shape (N,)
        Azimuthal angle in radians.
    xmin, xmax : float   – bounding box x extents
    ymin, ymax : float   – bounding box y extents
    zmin, zmax : float   – bounding box z extents

    Returns
    -------
    mask : np.ndarray, dtype=bool, shape (N,)
        True where the ray intersects the AABB.
    """
    x   = np.asarray(x,   dtype=float)
    y   = np.asarray(y,   dtype=float)
    z   = np.asarray(z,   dtype=float)
    mu  = np.asarray(mu,  dtype=float)
    phi = np.asarray(phi, dtype=float)

    # ------------------------------------------------------------------ #
    # Direction vectors from (mu, phi)                                     #
    # ------------------------------------------------------------------ #
    sin_theta = np.sqrt(np.clip(1.0 - mu**2, 0.0, 1.0))
    dx = sin_theta * np.cos(phi)
    dy = sin_theta * np.sin(phi)
    dz = mu.copy()

    # ------------------------------------------------------------------ #
    # Slab intersections for each axis                                     #
    # ------------------------------------------------------------------ #
    t_near = np.full(x.shape, -np.inf)
    t_far  = np.full(x.shape,  np.inf)

    for origin, direction, lo, hi in (
        (x, dx, xmin, xmax),
        (y, dy, ymin, ymax),
        (z, dz, zmin, zmax),
    ):
        # Parallel rays (direction ≈ 0)
        parallel = direction == 0.0
        inside   = parallel & (origin >= lo) & (origin <= hi)
        outside  = parallel & ~inside

        with np.errstate(divide="ignore", invalid="ignore"):
            t1 = np.where(~parallel, (lo - origin) / direction, np.nan)
            t2 = np.where(~parallel, (hi - origin) / direction, np.nan)

        # Ensure t1 is always the near hit and t2 is the far hit
        t_axis_near = np.minimum(t1, t2)
        t_axis_far  = np.maximum(t1, t2)

        # Parallel & outside: no intersection possible – push interval
        # to ±inf so the final test will fail for those rays.
        t_axis_near = np.where(outside,  np.inf, t_axis_near)
        t_axis_far  = np.where(outside, -np.inf, t_axis_far)

        # Parallel & inside: no constraint on t from this axis
        t_axis_near = np.where(inside, -np.inf, t_axis_near)
        t_axis_far  = np.where(inside,  np.inf, t_axis_far)

        t_near = np.maximum(t_near, t_axis_near)
        t_far  = np.minimum(t_far,  t_axis_far)

    # ------------------------------------------------------------------ #
    # A ray hits the box when the overlap is valid and not entirely        #
    # behind the ray origin (t_far >= 0).                                  #
    # ------------------------------------------------------------------ #
    mask = (t_near <= t_far) & (t_far >= 0.0)
    return mask


# ======================================================================= #
# Quick self-test                                                           #
# ======================================================================= #
if __name__ == "__main__":
    rng = np.random.default_rng(0)

    # Bounding box: unit cube [0,1]^3
    bbox = dict(xmin=0, xmax=1, ymin=0, ymax=1, zmin=0, zmax=1)

    # ── Test 1: rays fired from inside the box, all directions ──────── #
    N = 1_000
    x = rng.uniform(0, 1, N)
    y = rng.uniform(0, 1, N)
    z = rng.uniform(0, 1, N)
    mu  = rng.uniform(-1, 1, N)
    phi = rng.uniform(0, 2 * np.pi, N)

    mask = filter_rays_bbox(x, y, z, mu, phi, **bbox)
    assert mask.all(), "Rays starting inside the box must all pass through it."
    print("Test 1 passed: all rays from inside the box are selected.")

    # ── Test 2: rays aimed directly at the box from outside ─────────── #
    #   Origins far in front, directions pointing straight at the box centre.
    cx, cy, cz = 0.5, 0.5, 0.5
    x2 = rng.uniform(-10, -2, N)
    y2 = rng.uniform(0, 1, N)
    z2 = rng.uniform(0, 1, N)
    # Direction toward centre
    vec = np.column_stack([cx - x2, cy - y2, cz - z2])
    norm = np.linalg.norm(vec, axis=1, keepdims=True)
    vec /= norm
    dx2, dy2, dz2 = vec[:, 0], vec[:, 1], vec[:, 2]
    mu2  = dz2
    phi2 = np.arctan2(dy2, dx2)

    mask2 = filter_rays_bbox(x2, y2, z2, mu2, phi2, **bbox)
    assert mask2.all(), "Rays aimed at box centre from outside must all hit."
    print("Test 2 passed: all rays aimed at the box centre are selected.")

    # ── Test 3: rays clearly missing the box ────────────────────────── #
    x3 = rng.uniform(-10, -2, N)
    y3 = rng.uniform(-10, -2, N)   # far outside in y too
    z3 = rng.uniform(0, 1, N)
    mu3  = np.zeros(N)             # horizontal
    phi3 = np.zeros(N)             # pointing in +x, y-offset keeps them away

    mask3 = filter_rays_bbox(x3, y3, z3, mu3, phi3, **bbox)
    assert not mask3.any(), "Rays well outside the box must all be rejected."
    print("Test 3 passed: all rays missing the box are rejected.")

    # ── Test 4: mixed batch ──────────────────────────────────────────── #
    x4   = np.array([ 0.5,  -5.0])
    y4   = np.array([ 0.5,   0.5])
    z4   = np.array([ 0.5,   0.5])
    mu4  = np.array([ 0.0,   0.0])        # horizontal
    phi4 = np.array([ 0.0,   np.pi])      # +x and −x

    mask4 = filter_rays_bbox(x4, y4, z4, mu4, phi4, **bbox)
    assert mask4[0], "Ray 0 (inside, going +x) should hit the box."
    # Ray 1 starts at x=-5 pointing in -x direction (phi=pi → dx=cos(pi)=-1),
    # so it moves further from the box and should NOT hit.
    assert not mask4[1], "Ray at x=-5 pointing away in -x should not hit box."
    print("Test 4 passed: mixed hit/miss correctly identified.")

    print("\nAll tests passed.")

Test 1 passed: all rays from inside the box are selected.
Test 2 passed: all rays aimed at the box centre are selected.
Test 3 passed: all rays missing the box are rejected.
Test 4 passed: mixed hit/miss correctly identified.

All tests passed.


In [12]:
h = 1400
wavelength = 0.67

bbox = dict(xmin=0, xmax=17600, ymin=0, ymax=17600, zmin=0, zmax=h)

obs = at3d.containers.SensorsDict()
for c in imagery.camera.data:
    img_tmp = imagery.sel(camera=c)
    x = np.ravel(img_tmp.x.data)
    y = np.ravel(img_tmp.y.data)
    z = np.ravel(img_tmp.z.data)
    mu = np.ravel(img_tmp.mu.data)
    phi = np.ravel(img_tmp.phi.data)
    cond = filter_rays_bbox(x, y, z, mu, phi, **bbox)
    x, y, z = register_to_height(x, y, z, mu, phi, h)
    mu0 = np.cos(np.radians(origin.SolarZenith.data))
    I = np.ravel(img_tmp.brf.data) / np.pi
    sensor = at3d.sensor.make_sensor_dataset(
        x[cond]/1000,
        y[cond]/1000,
        z[cond]/1000,
        mu[cond],
        phi[cond],
        'I',
        wavelength,
        fill_ray_variables = True
    )
    sensor['I'] = (['nrays'], I[cond])
    sensor['I'].attrs['description'] = 'I = BRF_MISR / pi = I /  (mu0 * F_0)'
    sensor['brf'] = (['nrays'], np.ravel(img_tmp.brf.data)[cond])
    sensor['brf'].attrs['description'] = 'MISR BRF'
    sensor['mu0'] = mu0
    sensor['mu0'].attrs['long_name'] = 'cosine of the solar zenith angle'
    sensor['SAA'] = origin.SolarAzimuth.data
    sensor['SAA'].attrs['description'] = origin.SolarAzimuth.description
    sensor['SAA'].attrs['units'] = 'degrees'
    sensor.attrs['name'] = c
    sensor.attrs['wind_correction_u'] = u
    sensor.attrs['wind_correction_v'] = v
    obs.add_sensor('MISR', sensor)

outfilename = f'AT3D_OBS_bbox_O{orbit:06d}_{imagery.regionID}_H{int(h):d}_U{str(u).replace(".","p")}_V{str(v).replace(".","p")}.nc'
save_sensor_dict(PROJECT_ROOT / 'data' / 'V12' / outfilename, obs)

Saved sensors to: /data/gdi/b/johntl3/LundstromPub2025/data/V12/AT3D_OBS_bbox_O105254_P117_R00275_X17950487_18003012_Y0193187_0245712_H1400_U-0p2_V2p3.nc


## Plot

In [13]:
# plt.style.use('seaborn-v0_8-talk')
# plot_kwargs = {
#     'cmap':'gray',
#     'vmin':0,
#     'vmax':0.4,
#     'levels':np.linspace(0, 0.4, 50)
# }

# f, ax = plt.subplots(1, 9, figsize=(14, 1.3), sharex=True, sharey=True)
# plt.subplots_adjust(wspace=0.0, hspace=0.0)

# for i, cam in enumerate(obs['MISR']['sensor_list']):
#     c = ax[i].tricontourf(cam.ray_x.data, cam.ray_y.data, cam.brf.data, **plot_kwargs)
# plt.colorbar(c, ax=ax[:])

# box_kwargs = {'ls':'--', 'lw':0.7, 'c':'r'}
# ticks = [0.0, 17.6]
# for a in ax.flat:
#     a.plot([0, 0], [0, 17.6], **box_kwargs)
#     a.plot([0, 17.6], [0, 0], **box_kwargs)
#     a.plot([17.6, 17.6], [0, 17.6], **box_kwargs)
#     a.plot([0, 17.6], [17.6, 17.6], **box_kwargs)
#     # a.set_xticks(ticks)
#     # a.set_xticklabels(ticks, rotation=45)
#     # a.set_yticks(ticks)
#     # a.set_xlim([0,17.6])
#     # a.set_ylim([0,17.6])